Lê os 12 CSVs do volume `voebem.bronze.arquivos/vra/` e materializa `voebem.bronze.vra`

Regras da camada bronze
- **sem tipagem** - tudo como string, exatamente como veio do arquivo
- **nada de filtro** - nenhuma linha é descartada
- **colunas de auditoria** - de qual arquivo veio e quando foi ingerido
- **idempotente** - rodar duas vezes não duplica


In [0]:
from pyspark.sql import functions as F

CAMINHO = "/Volumes/voebem/bronze/arquivos/vra/*.csv"
TABELA = "voebem.bronze.vra"

In [0]:
bruto = (
    spark.read.format("csv")
  .option("header", True)
  .option("sep", ";")
  .option("skipRows", 1)
  .option("quote", '"')
  .option("escape", '"')
  .option("encoding", "UTF-8")
  .option("mode", "PERMISSIVE") # bronze não descarta nenhuma linha
  .load(CAMINHO)
  )

print("colunas lidas do arquivo:")
for c in bruto.columns:
  print(f"  {c!r}")

# Nomes de coluna: o Delta não aceita espaço

`ICAO Empresa Aérea` é um nome de coluna válido em CSV e inválido em Delta — o caractere de espaço (` `) está na lista de caracteres proibidos pela [especificação de nomenclatura de colunas do Delta Lake](https://docs.delta.io/latest/delta-batch.html#column-naming): `. ; { } ( ) \n \r \t = ,`

Portanto, antes de materializar a tabela Delta na camada bronze, **precisamos normalizar os nomes das colunas**, substituindo espaços por underscores (`_`) e removendo caracteres inválidos. A regra adotada nesta camada bronze é simples:

| Caractere original | Ação tomada |
|---|---|
| Espaço (` `)        | Substituir por `_` |
| Acentos (`á`, `ç`, …) | Remover (ex.: `Empresas Aérea` → `Empresas_Aerea`) |
| Demais proibidos      | Remover |

> **Observação:** mesmo após a normalização, nenhum dado é alterado — apenas os **nomes das colunas** mudam. Os valores continuam como strings brutas, conforme a regra da bronze de "sem tipagem".

In [0]:
import unicodedata

# --- 1. Função de normalização de nome de coluna ---
# Regra bronze: minúsculo, espaços -> _, acentos removidos, demais
# caracteres proibidos pelo Delta removidos.
_PROIBIDOS = set(".;{}()\n\r\t=,")

def normalizar_nome(col: str) -> str:
    # remove acentos (á -> a, ç -> c, etc.)
    sem_acento = unicodedata.normalize("NFKD", col)
    sem_acento = "".join(
        c for c in sem_acento if not unicodedata.combining(c)
    )
    # minúsculo + espaços -> _
    nome = sem_acento.lower().replace(" ", "_")
    # remove demais caracteres proibidos pelo Delta
    nome = "".join(c for c in nome if c not in _PROIBIDOS)
    return nome

# --- 2. Mapeamento original -> normalizado ---
renome = {c: normalizar_nome(c) for c in bruto.columns}

print("Mapeamento de colunas:")
for original, novo in renome.items():
    marca = "  ✓" if original != novo else ""
    print(f'  {original!r:40s} -> {novo!r} {marca}')

# --- 3. Aplica o rename ---
bruto = bruto.toDF(*[renome[c] for c in bruto.columns])

In [0]:
# Adiciona colunas de auditoria
bronze = bruto.withColumn("_arquivo_origem", F.col("_metadata.file_name")).withColumn("_ingerido_em", F.current_timestamp())

display(bronze.limit(5))

# Escrita idempotente

## Estratégia: Full refresh determinístico

A tabela bronze é materializada com `.mode("overwrite")`, reprocessando **todos** os arquivos do volume a cada execução.

### Por que overwrite e não append com deduplicação?

| Critério | Full refresh (overwrite) | Append + dedup |
|---|---|---|
| **Simplicidade** | ✅ Uma única operação: lê tudo, escreve tudo | ❌ Requer lógica de merge/dedup baseada em chave composta |
| **Velocidade** | ✅ Rápido para datasets pequenos/médios (< 10 GB) | ❌ Merge é mais lento que overwrite em volumes pequenos |
| **Garantia de consistência** | ✅ O estado final reflete **exatamente** os arquivos atuais | ⚠️ Requer identificar chave primária correta (não trivial neste dataset) |
| **Idempotência** | ✅ Rodar 2x produz o mesmo resultado | ✅ Se a dedup estiver correta, também é idempotente |
| **Custo** | ✅ Reescreve apenas o necessário (mesmos dados = mesmos arquivos Parquet) | ⚠️ Merge gera mais I/O e overhead |

### Quando mudar para append?

Quando o volume de dados crescer significativamente (> 100 GB) ou quando **novos arquivos chegarem incrementalmente** (ex: um CSV novo por dia), a estratégia de append com merge se torna mais eficiente. Nesse cenário:

```python
# Exemplo futuro (não implementado aqui):
bronze.write \
  .format("delta") \
  .mode("append") \
  .option("mergeSchema", "true") \
  .saveAsTable(TABELA)

# Seguido de deduplicação:
spark.sql(f"""
  MERGE INTO {TABELA} target
  USING (
    SELECT * FROM (
      SELECT *, ROW_NUMBER() OVER (
        PARTITION BY icao_empresa_aerea, numero_voo, partida_prevista 
        ORDER BY _ingerido_em DESC
      ) as rn
      FROM {TABELA}
    ) WHERE rn = 1
  ) source
  ON target.id = source.id  -- requer chave primária
  WHEN MATCHED THEN UPDATE SET *
  WHEN NOT MATCHED THEN INSERT *
""")
```

Para este dataset de 12 CSVs com ~50K linhas, **overwrite é a escolha certa**.

In [0]:
# Materializa a tabela Delta com full refresh
bronze.write \
  .format("delta") \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable(TABELA)

print(f"✅ Tabela {TABELA} materializada com sucesso!")
print(f"   Total de registros: {spark.table(TABELA).count():,}")

In [0]:
spark.sql(f"""
    COMMENT ON TABLE {TABELA} IS
    'Bronze - VRA (Voo Regular Ativo) da ANAC, 12 meses (ago/2025 a jul/2026).
        Dado bruto: todas as colunas string, nenhuma linha descartada.
        Carga full refresh idempotente a partir de /Volumes/voebem/bronze/arquivos/vra/.'
    """)

In [0]:
display(
    spark.sql(f"""
        SELECT _arquivo_origem, count(*) as linhas, MAX (_ingerido_em) as data_ingerido
        FROM {TABELA}
        GROUP BY _arquivo_origem
        ORDER BY _arquivo_origem
        """)
)